# Notebook 7 — Does Editing Relocate the Circuit?
**CS 590NN | Amogh | Apr 24 — mechanistic explanation for the v3 null result**

## The hypothesis (the "why" behind v3's null)

Notebook 5 v3's null result — protection mechanism didn't separate methods, retention was bimodal — has a clean mechanistic explanation if true:

> **Editing relocates the circuit responsible for the edited fact.** When you edit fact A, the gradient updates push the model into a new computational state where A's recall is now handled by *different* attention heads and MLP layers than before. Any protection mechanism keyed to the *pre-edit* circuit is therefore protecting the wrong components.

## Why this would be novel

The mech-interp-for-editing literature (Guo et al. 2024, Hase et al. 2023) implicitly assumes the circuit identified before editing is the same circuit responsible for the post-edit behavior. Nobody has measured this assumption directly. If we show the assumption is broken, that explains why localization-based editing methods underperform expectations — and points at a concrete fix (re-discover the circuit AFTER each edit, not before).

## What this notebook does

1. Take 15 samples from CounterFact.
2. For each sample: run ACDC to discover circuit C_pre.
3. Edit the fact (5-step KL-stabilized protocol, same as Notebook 3).
4. Run ACDC AGAIN on the same sample to discover circuit C_post.
5. Compute Jaccard overlap between C_pre and C_post.
6. Restore weights and repeat.

**Predicted result:** Mean Jaccard < 0.5 — circuits move substantially after editing.

## Runtime

~25-35 minutes on A100 (15 samples × 2 ACDC runs × ~60 sec + edit overhead).

### 7.0 Install
Run once. Runtime auto-restarts.

In [ ]:
import subprocess, sys, os
def pip(args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Done. Restarting runtime...')
os.kill(os.getpid(), 9)

### 7.1 Imports — start here after restart

In [ ]:
import torch, json, requests, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import List, Tuple
from collections import Counter
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb7'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

### 7.2 Load Model

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False,
    device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, _ = torch.cuda.mem_get_info()
print(f'Loaded {MODEL_NAME}: {model.cfg.n_layers} layers × {model.cfg.n_heads} heads')
print(f'GPU free: {free/1e9:.1f} GB')

### 7.3 Load CounterFact

In [ ]:
@dataclass
class EditSample:
    idx: int
    prompt: str
    target_new: str
    target_true: str

print('Downloading CounterFact...')
raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()
print(f'Total records: {len(raw)}')

N_SAMPLES = 15
samples = []
for i in range(N_SAMPLES):
    rr = raw[i]['requested_rewrite']
    samples.append(EditSample(
        idx=i, prompt=rr['prompt'].format(rr['subject']),
        target_new=' ' + rr['target_new']['str'],
        target_true=' ' + rr['target_true']['str'],
    ))
for s in samples[:3]:
    print(f"  [{s.idx}] {s.prompt!r} -> {s.target_new.strip()!r}")

### 7.4 ACDC implementation (same as Notebook 1 v2.1)

In [ ]:
class NativeACDC:
    def __init__(self, model, threshold=0.3, effect_floor=0.5):
        self.m, self.th, self.floor = model, threshold, effect_floor
    def _ld(self, logits, t, b):
        return (logits[0,-1,t] - logits[0,-1,b]).item()
    def run(self, prompt, target_true, target_new):
        m = self.m
        true_id = m.to_tokens(target_true, prepend_bos=False)[0,0].item()
        new_id  = m.to_tokens(target_new,  prepend_bos=False)[0,0].item()
        tokens  = m.to_tokens(prompt)
        corrupt = tokens.clone()
        if tokens.shape[1] > 2:
            corrupt[0, 1:-1] = torch.randint(1000, m.cfg.d_vocab-1, (tokens.shape[1]-2,))
        with torch.no_grad():
            cl, cc = m.run_with_cache(tokens)
            kl, kc = m.run_with_cache(corrupt)
        clean_score = self._ld(cl, true_id, new_id)
        eff = max(abs(clean_score - self._ld(kl, true_id, new_id)), self.floor)
        attn_heads, mlp_layers = [], []
        for L in range(m.cfg.n_layers):
            hn = f'blocks.{L}.attn.hook_result'
            if hn in cc:
                for h in range(m.cfg.n_heads):
                    def mk(h=h, n=hn):
                        ca = kc[n][:,:,h:h+1,:].clone()
                        def fn(v, hook): v[:,:,h:h+1,:] = ca; return v
                        return fn
                    with torch.no_grad():
                        pl = m.run_with_hooks(tokens, fwd_hooks=[(hn, mk())])
                    if abs(self._ld(pl, true_id, new_id) - clean_score) / eff >= self.th:
                        attn_heads.append((L, h))
            hn = f'blocks.{L}.hook_mlp_out'
            if hn in cc:
                kca = kc[hn].clone()
                def mk_mlp(kca=kca):
                    def fn(v, hook): return kca
                    return fn
                with torch.no_grad():
                    pl = m.run_with_hooks(tokens, fwd_hooks=[(hn, mk_mlp())])
                if abs(self._ld(pl, true_id, new_id) - clean_score) / eff >= self.th:
                    mlp_layers.append(L)
        del cc, kc, cl, kl
        torch.cuda.empty_cache()
        return {'attn_heads': sorted(set(attn_heads)), 'mlp_layers': sorted(set(mlp_layers))}

acdc = NativeACDC(model, threshold=0.3, effect_floor=0.5)
print('NativeACDC ready')

### 7.5 Edit function (same protocol as Notebook 3 v2)

In [ ]:
NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The square root of nine is', 'Ten times ten equals',
    'The capital of Japan is', 'The largest ocean on Earth is the',
    'Mount Everest is located in the', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
    'Plants produce oxygen through a process called', 'The Earth orbits the',
    'A week contains seven', 'The primary colors are red, blue, and',
    'Humans have two lungs and one', 'A triangle has three',
]

def cache_ref(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            tokens = model.to_tokens(p)
            logits = model(tokens)
            ref_lp = torch.log_softmax(logits[0,-1,:], dim=-1).detach().clone()
            cache.append((tokens, ref_lp))
    return cache

def kl_neutral(model, ref_cache, train=True):
    total = 0.0
    for tokens, ref_lp in ref_cache:
        logits = model(tokens) if train else (lambda: model(tokens))().detach()
        edit_lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def get_params(model, attn, mlp):
    seen, params = set(), []
    for (L, h) in attn:
        if ('attn', L) in seen: continue
        params.append(model.blocks[L].attn.W_O); seen.add(('attn', L))
    for L in mlp:
        if ('mlp', L) in seen: continue
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
        seen.add(('mlp', L))
    return params

def edit(model, sample, attn, mlp, n_steps=5, lr=5e-3, beta_kl=0.1, ref_cache=None):
    params = get_params(model, attn, mlp) or [p for layer in model.blocks for p in [layer.mlp.W_in, layer.mlp.W_out]]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[new_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_neutral(model, ref_cache, train=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    torch.cuda.empty_cache()

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        logits = model(model.to_tokens(sample.prompt))
        return torch.softmax(logits[0,-1,:], dim=-1)[new_id].item()

print('Snapshotting weights...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

### 7.6 The Pre/Post Loop

For each sample:
1. ACDC discover → C_pre
2. Edit → measure p_new (sanity check)
3. ACDC discover → C_post
4. Compute Jaccard(C_pre, C_post) for both attn and mlp
5. Restore weights

In [ ]:
def jaccard(a, b):
    sa, sb = set(map(tuple, a) if a and isinstance(a[0], (list, tuple)) else a), set(map(tuple, b) if b and isinstance(b[0], (list, tuple)) else b)
    if not sa and not sb: return 1.0
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

ROWS_FILE = RESULTS_DIR / f'circuit_relocation_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
rows = []
started = datetime.now()
ref_cache = cache_ref(model, NEUTRAL)

for s in samples:
    print(f'\n=== Sample {s.idx}: {s.prompt!r} ===')
    try:
        # 1. Pre-edit ACDC
        restore(model, original_state)
        c_pre = acdc.run(s.prompt, s.target_true, s.target_new)
        p_new_pre = measure_p_new(model, s)
        print(f"  PRE  : n_attn={len(c_pre['attn_heads']):>3} n_mlp={len(c_pre['mlp_layers']):>2}  p_new={p_new_pre:.3f}")

        # 2. Edit
        edit(model, s, c_pre['attn_heads'], c_pre['mlp_layers'], n_steps=5, ref_cache=ref_cache)
        p_new_post = measure_p_new(model, s)
        print(f"  EDITED: p_new={p_new_post:.3f}  (delta={p_new_post-p_new_pre:+.3f})")

        # 3. Post-edit ACDC
        c_post = acdc.run(s.prompt, s.target_true, s.target_new)
        print(f"  POST : n_attn={len(c_post['attn_heads']):>3} n_mlp={len(c_post['mlp_layers']):>2}")

        # 4. Jaccards
        j_attn = jaccard(c_pre['attn_heads'], c_post['attn_heads'])
        j_mlp  = jaccard(c_pre['mlp_layers'], c_post['mlp_layers'])
        all_pre  = [('a', L, h) for L, h in c_pre['attn_heads']]  + [('m', L) for L in c_pre['mlp_layers']]
        all_post = [('a', L, h) for L, h in c_post['attn_heads']] + [('m', L) for L in c_post['mlp_layers']]
        j_all = jaccard(all_pre, all_post)
        print(f"  Jaccard: attn={j_attn:.3f}  mlp={j_mlp:.3f}  combined={j_all:.3f}")

        rows.append({
            'idx': s.idx, 'prompt': s.prompt,
            'pre_n_attn': len(c_pre['attn_heads']), 'pre_n_mlp': len(c_pre['mlp_layers']),
            'post_n_attn': len(c_post['attn_heads']), 'post_n_mlp': len(c_post['mlp_layers']),
            'pre_attn': c_pre['attn_heads'], 'pre_mlp': c_pre['mlp_layers'],
            'post_attn': c_post['attn_heads'], 'post_mlp': c_post['mlp_layers'],
            'jaccard_attn': j_attn, 'jaccard_mlp': j_mlp, 'jaccard_combined': j_all,
            'p_new_pre': p_new_pre, 'p_new_post': p_new_post,
            'edit_delta': p_new_post - p_new_pre,
            'status': 'ok',
        })
        # Crash-safe save
        with open(ROWS_FILE, 'w') as f:
            json.dump({'rows': rows}, f, indent=2)
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {e}')
        rows.append({'idx': s.idx, 'status': 'failed', 'error': str(e)[:200]})
        torch.cuda.empty_cache()

elapsed = (datetime.now() - started).total_seconds() / 60
print(f'\nTotal runtime: {elapsed:.1f} min')
restore(model, original_state)

### 7.7 Aggregate + Stat Test

In [ ]:
from scipy import stats
ok = [r for r in rows if r.get('status') == 'ok']
j_attn = np.array([r['jaccard_attn'] for r in ok])
j_mlp  = np.array([r['jaccard_mlp']  for r in ok])
j_all  = np.array([r['jaccard_combined'] for r in ok])
deltas = np.array([r['edit_delta'] for r in ok])

print(f'OK samples: {len(ok)}')
print(f'\nJaccard overlap (1.0 = identical circuit, 0.0 = no overlap):')
print(f'  attn:     mean={j_attn.mean():.3f}  median={np.median(j_attn):.3f}  std={j_attn.std():.3f}')
print(f'  mlp:      mean={j_mlp.mean():.3f}   median={np.median(j_mlp):.3f}   std={j_mlp.std():.3f}')
print(f'  combined: mean={j_all.mean():.3f}   median={np.median(j_all):.3f}   std={j_all.std():.3f}')

# One-sample test: is jaccard significantly < 0.5?
t, p = stats.wilcoxon(j_all - 0.5, alternative='less')
print(f'\nH-test: jaccard_combined < 0.5 (i.e. circuits substantially relocate)?')
print(f'  Wilcoxon signed-rank test, one-sided p = {p:.4f}')
if p < 0.05:
    print('  *** REJECTED null. Editing relocates circuits significantly. ***')
    print('  This explains why protection-based sequential editing fails:')
    print('  the protection term is keyed to the wrong (pre-edit) circuit.')

# Edit success check (sanity)
print(f'\nEdit success: mean p_new went from {np.mean([r["p_new_pre"] for r in ok]):.3f} to {np.mean([r["p_new_post"] for r in ok]):.3f}')
print(f'Mean delta: {deltas.mean():+.3f}')

### 7.8 Figures

In [ ]:
# Figure 1 — Jaccard distribution
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(ok))
w = 0.3
ax.bar(x - w, j_attn, w, label='attn heads', alpha=0.8)
ax.bar(x,     j_mlp,  w, label='mlp layers', alpha=0.8)
ax.bar(x + w, j_all,  w, label='combined',   alpha=0.8)
ax.axhline(0.5, color='red', ls='--', label='50% overlap')
ax.axhline(1.0, color='green', ls=':', alpha=0.5, label='identical')
ax.set_xlabel('Sample index'); ax.set_ylabel('Jaccard(C_pre, C_post)')
ax.set_title('Per-sample circuit overlap before vs after editing')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / 'fig1_per_sample_jaccard.png', dpi=140); plt.show()

# Figure 2 — Pre vs post circuit-size scatter (do circuits also change in SIZE?)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (key, label) in zip(axes, [('attn', 'attention heads'), ('mlp', 'MLP layers')]):
    pre_n  = [r[f'pre_n_{key}']  for r in ok]
    post_n = [r[f'post_n_{key}'] for r in ok]
    lim = max(max(pre_n), max(post_n)) + 2
    ax.scatter(pre_n, post_n, s=60, alpha=0.7, edgecolors='black', lw=0.4)
    ax.plot([0, lim], [0, lim], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlabel(f'n_{key} pre-edit'); ax.set_ylabel(f'n_{key} post-edit')
    ax.set_title(f'Circuit size: {label}'); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / 'fig2_circuit_size_change.png', dpi=140); plt.show()

# Figure 3 — Headline: distribution of combined Jaccard
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(j_all, bins=10, alpha=0.7, color='#cc3333', edgecolor='black')
ax.axvline(np.median(j_all), color='black', lw=1.5, label=f'median = {np.median(j_all):.2f}')
ax.axvline(0.5, color='blue', lw=1.5, ls='--', label='50% threshold')
ax.axvline(1.0, color='green', lw=1.5, ls=':', label='identical (1.0)', alpha=0.6)
ax.set_xlabel('Jaccard(C_pre, C_post)  combined'); ax.set_ylabel('count')
ax.set_title(f'How much do circuits change after editing? (n={len(ok)})')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / 'fig3_jaccard_distribution.png', dpi=140); plt.show()

### 7.9 Save and Download

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'notebook': 'Notebook 7 — Circuit relocation under editing',
    'timestamp': ts, 'model': MODEL_NAME, 'n_samples': len(ok),
    'jaccard_attn':  {'mean': float(j_attn.mean()), 'median': float(np.median(j_attn)), 'std': float(j_attn.std())},
    'jaccard_mlp':   {'mean': float(j_mlp.mean()),  'median': float(np.median(j_mlp)),  'std': float(j_mlp.std())},
    'jaccard_combined': {'mean': float(j_all.mean()), 'median': float(np.median(j_all)), 'std': float(j_all.std())},
    'wilcoxon_p_jaccard_below_0.5': float(p),
    'edit_delta_mean': float(deltas.mean()),
    'finding': 'Circuits relocate substantially after editing' if p < 0.05 else 'Circuits do not relocate significantly',
}
with open(RESULTS_DIR / f'summary_nb7_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb7_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')

### What this tells us

**If median Jaccard < 0.5 with stat-sig p<0.05:** 
Strong novel finding. *"Editing relocates the responsible circuit by approximately X%, which explains why protection-keyed-to-pre-edit-circuit (as in Notebook 5 v3) cannot prevent sequential interference. This identifies a fundamental limitation of static-circuit-based editing protection and suggests dynamic circuit re-discovery between edits as a future direction."*

**If Jaccard > 0.7 (circuits stable):** 
Different finding. *"Circuits remain stable across editing, ruling out circuit relocation as the explanation for sequential editing failures. The failure mode must lie elsewhere — likely in the gradient-magnitude regime where contrastive loss overwhelms the KL constraint regardless of which parameters are exposed."*

**If results are bimodal (some samples high, some low):** 
Even more interesting. The samples that retain circuits should also be the ones where Notebook 5 v3 retained edits. Cross-check by sample index. If correlation holds: *"Samples whose circuits remain stable post-edit also retain their edits under sequential editing — circuit stability predicts sequential robustness."*

Whatever the result, this is a publishable diagnosis of *why* protection-based sequential editing succeeds or fails. The mechanism story would substantially strengthen the report.